# JobGuard AI: Exploratory Data Analysis & Leakage Auditing
Dataset: **EMSCAD (Real / Fake Job Posting Prediction)** (~17,880 postings, ~4.8% fraud).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.insert(0, "..")
from src.data.loader import load_raw_data
from src.data.cleaner import clean_job_postings
from src.features.leakage import create_leakage_free_split, check_leakage_between_splits

sns.set_theme(style="whitegrid")

## 1. Load Raw Dataset

In [ ]:
df_raw = load_raw_data("../data/raw/fake_job_postings.csv")
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Target distribution:\n{df_raw['fraudulent'].value_counts(normalize=True)}")

## 2. Missing Value Analysis by Class

In [ ]:
missing_summary = pd.DataFrame({
    'Legitimate': df_raw[df_raw['fraudulent'] == 0].isna().mean(),
    'Fraudulent': df_raw[df_raw['fraudulent'] == 1].isna().mean()
})
missing_summary['Diff (Fake - Real)'] = missing_summary['Fraudulent'] - missing_summary['Legitimate']
missing_summary.sort_values(by='Diff (Fake - Real)', ascending=False)

## 3. Cleaning and Deduplication

In [ ]:
df_clean = clean_job_postings(df_raw, drop_duplicates=True, create_combined_text=True)
print(f"Deduplicated shape: {df_clean.shape}")
print(f"Fraud rate post-deduplication: {df_clean['fraudulent'].mean():.2%}")

## 4. Leakage-Free Stratified Group Splitting

In [ ]:
train_df, test_df = create_leakage_free_split(df_clean, test_size=0.2, random_state=42)
leakage_check = check_leakage_between_splits(train_df, test_df)
print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")
print(f"Fingerprint Overlap: {leakage_check['fingerprint_overlap_count']}")
print(f"Leakage-Free Verified: {leakage_check['is_leakage_free']}")